# 07 · Other MRI sequences, and seed variance
Most remaining errors are gliomas acquired differently (T2/FLAIR, other scanners). Adding gliomas alone would teach "T2 means glioma",
so the added data must cover several classes in each sequence: the Feltrin collection has glioma, meningioma and normal brains in
T1, contrast T1 and T2 (no pituitary class). Pools: KB (Kaggle v2 + BRISC) and KBF (+ Feltrin), same twin removal, 384 px,
EfficientNetV2-S, two seeds. Choices on BRISC test; Kaggle test and a held-out Feltrin part are reported.

In [ ]:
!pip -q install imagehash
import os, glob, json, hashlib, time, random, math
import numpy as np, pandas as pd
from PIL import Image, ImageFilter
import imagehash, torch, timm
from sklearn.model_selection import StratifiedGroupKFold
SEED = 42; OUT = "/kaggle/working"; DEV = "cuda"
def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]; C2I = {c: i for i, c in enumerate(CLASSES)}; NT = C2I["notumor"]
NORM = {"glioma": "glioma", "glioma_tumor": "glioma", "meningioma": "meningioma", "meningioma_tumor": "meningioma",
        "notumor": "notumor", "no_tumor": "notumor", "pituitary": "pituitary", "pituitary_tumor": "pituitary"}
def find_dir(keys, name):
    m = [d for d in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if any(k in d for k in keys) and os.path.isdir(d)]
    assert m, (keys, name); return sorted(m, key=len)[0]
KAGK, BRK, SARK = ["masoudnickparvar", "brain-tumor-mri-dataset"], ["brisc"], ["sartaj", "brain-tumor-classification-mri"]
KAG_TR, KAG_TE = find_dir(KAGK, "Training"), find_dir(KAGK, "Testing")
BR = find_dir(BRK, "classification_task")
SAR_TR, SAR_TE = find_dir(SARK, "Training"), find_dir(SARK, "Testing")
assert len({KAG_TR, SAR_TR}) == 2
print(KAG_TR, BR, SAR_TR, sep="\n")
rows = []
def add(root, ds, split):
    for cls in sorted(os.listdir(root)):
        if cls not in NORM: continue
        for f in sorted(os.listdir(f"{root}/{cls}")):
            rows.append(dict(path=f"{root}/{cls}/{f}", ds=ds, split=split, label=NORM[cls]))
add(KAG_TR, "kaggle", "train"); add(KAG_TE, "kaggle", "test")
add(f"{BR}/train", "brisc", "train"); add(f"{BR}/test", "brisc", "test")
add(SAR_TR, "sartaj", "train"); add(SAR_TE, "sartaj", "train")      # SARTAJ is used for training only
FELK = ["fernando2rad", "brain-tumor-mri-images-17-classes"]
fel_dirs = [d for d in glob.glob("/kaggle/input/**/*NORMAL*", recursive=True) if os.path.isdir(d) and any(k in d for k in FELK)]
FEL = os.path.dirname(sorted(fel_dirs, key=len)[0]); print("feltrin", FEL, sorted(os.listdir(FEL)))
def fel_label(folder):
    f = folder.lower()
    if "normal" in f: return "notumor"
    if f.startswith("meningioma"): return "meningioma"
    if f.startswith("glioma"): return "glioma"
    return None                                     # neurocytoma, schwannoma, other lesions: not in our four classes
for folder in sorted(os.listdir(FEL)):
    lab = fel_label(folder)
    if lab is None or not os.path.isdir(f"{FEL}/{folder}"): continue
    seq = folder.split()[-1].upper()
    for f in sorted(os.listdir(f"{FEL}/{folder}")):
        rows.append(dict(path=f"{FEL}/{folder}/{f}", ds="feltrin", split="train", label=lab, seq=seq))
A = pd.DataFrame(rows); A["seq"] = A.seq.fillna("T1C+")
print(A.groupby(["ds", "split", "label"]).size().unstack()); print(A[A.ds == "feltrin"].groupby(["label", "seq"]).size())

In [ ]:
_POP = np.array([bin(k).count("1") for k in range(256)], np.uint8)
popcount = lambda x: _POP[x.view(np.uint8).reshape(-1, 8)].sum(1)
t0 = time.time(); md5s, ph, th, wh = [], [], [], []
for p in A.path:
    md5s.append(hashlib.md5(open(p, "rb").read()).hexdigest())
    im = Image.open(p); wh.append(im.size); im = im.convert("RGB").convert("L"); ph.append(int(str(imagehash.phash(im)), 16))
    a = np.asarray(im.resize((64, 64), Image.BILINEAR), np.float32).ravel(); a = (a - a.mean()) / (a.std() + 1e-6)
    th.append(a / np.sqrt(len(a)))
A["md5"] = md5s; A["is512"] = [w == 512 and h == 512 for w, h in wh]
ph = np.array(ph, np.uint64); th = np.stack(th); print(f"hashed {len(A)} in {time.time()-t0:.0f}s")

def best_twin(q_idx, ref_idx):
    ref_idx = np.asarray(ref_idx); ref_md5 = set(A.md5.values[ref_idx]); best = np.zeros(len(q_idx))
    for k, i in enumerate(q_idx):
        if A.md5.values[i] in ref_md5: best[k] = 1.0; continue
        cand = ref_idx[popcount(ph[i] ^ ph[ref_idx]) <= 10]
        if len(cand):
            c = float((th[cand] @ th[i]).max()); best[k] = c if c >= 0.90 else 0
    return best
tiers = lambda b: pd.cut(pd.Series(b), [-1, 1e-4, 0.95, 0.98, 2], right=False,
                         labels=["no twin", "similar", "very similar", "near-identical"]).values
TEST_IDX = np.nonzero((A.split == "test").values)[0]
cand_idx = np.nonzero((A.split == "train").values)[0]
# remove from the pool every training image that has a twin in either test set (checked from the training side)
b = best_twin(cand_idx, TEST_IDX)
removed = pd.crosstab(A.ds.values[cand_idx], b > 0)
print("pool images removed for having a twin in a test set:\n", removed)
POOL = cand_idx[b == 0]
pool = A.loc[POOL].reset_index()            # 'index' keeps the row in A
print(pool.groupby(["ds", "label"]).size().unstack())

In [ ]:
# near-duplicate groups inside the pool, for a group-split validation set
n = len(POOL); pairs = []
for k in range(n - 1):
    i = POOL[k]; d = popcount(ph[i] ^ ph[POOL[k + 1:]])
    for j in np.nonzero(d <= 10)[0] + k + 1:
        if float(th[POOL[j]] @ th[i]) >= 0.90 or md5s[i] == md5s[POOL[j]]: pairs.append((k, int(j)))
parent = list(range(n))
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i, j in pairs: parent[find(i)] = find(j)
pool["group"] = [find(i) for i in range(n)]

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
SIZE = 384
def crop_brain(img, thr=12, pad=6):
    g = np.asarray(img.convert("L").filter(ImageFilter.GaussianBlur(2))); ys, xs = np.nonzero(g > thr)
    if len(xs) < 100: return img
    return img.crop((max(xs.min() - pad, 0), max(ys.min() - pad, 0), min(xs.max() + pad, g.shape[1]), min(ys.max() + pad, g.shape[0])))
CACHE = {}
def load(p):
    if p not in CACHE:
        im = crop_brain(Image.open(p).convert("RGB")); w, h = im.size; s = SIZE / max(w, h)
        im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)
        c = Image.new("RGB", (SIZE, SIZE)); c.paste(im, ((SIZE - im.width) // 2, (SIZE - im.height) // 2)); CACHE[p] = c
    return CACHE[p]
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
TF_TR = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(), T.RandomRotation(12),
                   T.ColorJitter(brightness=0.25, contrast=0.25), T.RandomGrayscale(0.1), T.ToTensor(), T.Normalize(MEAN, STD)])
TF_TE = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
class MRI(Dataset):
    def __init__(self, f, tf): self.p, self.y, self.tf = list(f.path), [C2I[l] for l in f.label], tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(load(self.p[i])), self.y[i]
def loader(f, train, bs=24):
    return DataLoader(MRI(f, TF_TR if train else TF_TE), batch_size=bs if train else 64, shuffle=train,
                      num_workers=4, pin_memory=True, drop_last=train)
t0 = time.time()
for p in pd.concat([pool.path, A.loc[TEST_IDX].path]): load(p)
print("cached", len(CACHE), f"in {time.time()-t0:.0f}s")

@torch.no_grad()
def predict(model, f, tta=True):
    model.eval(); out = []
    for x, _ in loader(f, False):
        x = x.to(DEV)
        with autocast("cuda", dtype=torch.float16):
            p = model(x).float().softmax(1)
            if tta: p = (p + model(torch.flip(x, dims=[3])).float().softmax(1)) / 2
        out.append(p.cpu())
    return torch.cat(out).numpy()

def train_model(name, epochs=16, lr=3e-4):
    seed_all(SEED)
    model = timm.create_model(name, pretrained=True, num_classes=4, drop_path_rate=0.1).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    tl = loader(train_df, True); steps = epochs * len(tl); warm = len(tl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, s / steps))))
    crit = torch.nn.CrossEntropyLoss(label_smoothing=0.1); scaler = GradScaler(); best, state = -1, None
    yv = val_df.label.map(C2I).values
    for ep in range(epochs):
        model.train(); t0 = time.time(); tot = 0
        for x, y in tl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with autocast("cuda", dtype=torch.float16): loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        pv = predict(model, val_df, tta=False); acc = (pv.argmax(1) == yv).mean()
        # macro accuracy over sources: the smallest source must not be drowned by the largest
        src_acc = pd.Series(pv.argmax(1) == yv).groupby(val_df.ds.values).mean()
        print(f"ep{ep:2d} loss {tot/len(tl):.3f} val {100*acc:.2f}%  by source {dict((100*src_acc).round(1))} ({time.time()-t0:.0f}s)")
        score = src_acc.mean()
        if score > best: best, state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state); torch.save(state, f"{OUT}/{name}_{TAG}.pt"); return model

In [ ]:
# Feltrin: 20% of its near-duplicate groups become a third test set (images from the same patient may still be shared,
# because the dataset has no patient ids; the group split removes only images with a twin)
fel = pool[pool.ds == "feltrin"]
g = np.array(sorted(fel.group.unique())); rng = np.random.default_rng(0); fel_test_groups = set(rng.choice(g, size=len(g) // 5, replace=False))
fel_test = pool[(pool.ds == "feltrin") & pool.group.isin(fel_test_groups)].reset_index(drop=True)
pool_tr = pool[~((pool.ds == "feltrin") & pool.group.isin(fel_test_groups))].reset_index(drop=True)
print("feltrin test", fel_test.groupby(["label", "seq"]).size().to_dict())
test = A.loc[TEST_IDX].reset_index(); y = test.label.map(C2I).values
dev, fin = (test.ds == "brisc").values, (test.ds == "kaggle").values
yf = fel_test.label.map(C2I).values

def run(sources, seed, tag):
    global train_df, val_df, TAG, SEED
    sub = pool_tr[pool_tr.ds.isin(sources)].reset_index(drop=True); sub["strat"] = sub.label + "_" + sub.ds
    tr, va = next(StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=42).split(sub, sub.strat, sub.group))
    train_df, val_df = sub.iloc[tr].reset_index(drop=True), sub.iloc[va].reset_index(drop=True)
    TAG, SEED = tag, seed; t0 = time.time()
    m = train_model("tf_efficientnetv2_s.in21k_ft_in1k", epochs=16)
    p, pf = predict(m, test), predict(m, fel_test); del m; torch.cuda.empty_cache()
    np.save(f"{OUT}/probs_{tag}.npy", p); np.save(f"{OUT}/probs_fel_{tag}.npy", pf)
    print(f"{tag}: train {len(train_df)}  BRISC {100*(p.argmax(1)[dev]==y[dev]).mean():.2f}  Kaggle {100*(p.argmax(1)[fin]==y[fin]).mean():.2f}"
          f"  Feltrin {100*(pf.argmax(1)==yf).mean():.2f}  ({(time.time()-t0)/60:.0f} min)")
    return p, pf

P = {}
for sources, seed, tag in [(["kaggle", "brisc", "feltrin"], 42, "KBF_s42"), (["kaggle", "brisc"], 7, "KB_s7"), (["kaggle", "brisc", "feltrin"], 7, "KBF_s7")]:
    P[tag] = run(sources, seed, tag)

def decide(p, t):
    yh = p.argmax(1).copy(); tum = p.copy(); tum[:, NT] = -1; low = (yh == NT) & (p[:, NT] < t)
    yh[low] = tum[low].argmax(1); return yh
def pick_threshold(p):
    rows = []
    for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
        yh = decide(p, t); rows.append(dict(t=t, acc=100 * (yh[dev] == y[dev]).mean(), missed=int(((yh == NT) & (y != NT) & dev).sum())))
    r = pd.DataFrame(rows); ok = r[r.acc >= r.acc.max() - 0.5]; return float(ok.sort_values(["missed", "t"]).iloc[0].t)
def report(yh, mask, f=None):
    f = (test if f is None else f)[mask] if mask is not None else f
    yy = f.label.map(C2I).values; ok = yh == yy; pred = np.array(CLASSES)[yh]
    r = dict(n=len(f), acc=round(100 * ok.mean(), 2), by_class=(100 * pd.Series(ok).groupby(f.label.values).mean()).round(1).to_dict(),
             missed_tumors=int(((pred == "notumor") & (f.label.values != "notumor")).sum()),
             false_alarms=int(((pred != "notumor") & (f.label.values == "notumor")).sum()))
    if "is512" in f and (f.label == "glioma").any():
        g = (f.label.values == "glioma") & ~f.is512.values; r["glioma_other_sources"] = round(100 * ok[g].mean(), 2) if g.any() else None
    if "seq" in f: r["by_seq"] = (100 * pd.Series(ok).groupby(f.seq.values).mean()).round(1).to_dict()
    return r

R = {}
for tag, (p, pf) in P.items():
    T = pick_threshold(p)
    R[tag] = dict(threshold=T, brisc_dev=report(decide(p, T)[dev], dev), kaggle=report(decide(p, T)[fin], fin),
                  kaggle_argmax=report(p.argmax(1)[fin], fin), feltrin=report(decide(pf, T), None, fel_test))
# two-seed ensembles
for name, tags in [("KBF_2seeds", ["KBF_s42", "KBF_s7"])]:
    p = np.mean([P[t][0] for t in tags], 0); pf = np.mean([P[t][1] for t in tags], 0); T = pick_threshold(p)
    R[name] = dict(threshold=T, brisc_dev=report(decide(p, T)[dev], dev), kaggle=report(decide(p, T)[fin], fin),
                   feltrin=report(decide(pf, T), None, fel_test))
    P[name] = (p, pf)

# abstention on the best configuration by BRISC dev accuracy: confidence threshold chosen on BRISC for 99.5% accuracy
best = max(R, key=lambda k: R[k]["brisc_dev"]["acc"]); p, pf = P[best]; T = R[best]["threshold"]
conf = p.max(1); yh = decide(p, T); ABST = {}
for target in [0.99, 0.995]:
    c_dev = np.sort(conf[dev])[::-1]; ok_dev = (yh[dev] == y[dev])[np.argsort(-conf[dev])]
    cum = np.cumsum(ok_dev) / np.arange(1, dev.sum() + 1); idx = np.nonzero(cum >= target)[0]
    thr = float(c_dev[idx.max()]) if len(idx) else 1.0
    ans = conf >= thr
    ABST[str(target)] = dict(conf_threshold=round(thr, 4),
        kaggle_coverage=round(100 * ans[fin].mean(), 2), kaggle_acc_answered=round(100 * (yh[fin & ans] == y[fin & ans]).mean(), 2),
        kaggle_missed_answered=int(((yh == NT) & (y != NT) & fin & ans).sum()),
        brisc_coverage=round(100 * ans[dev].mean(), 2), brisc_acc_answered=round(100 * (yh[dev & ans] == y[dev & ans]).mean(), 2))
R["best_by_dev"] = best; R["abstention"] = ABST
print(json.dumps(R, indent=1, default=str))
json.dump(R, open(f"{OUT}/results_07.json", "w"), indent=1, default=str)